# Annotate spots

Draws the spot polygons of `data/<GROUP>/labels/spots` on the operculum ROIs, with a number next to each spot and the spot count in the corner, and saves them to `images/spot_annotations/`.

In [12]:
import cv2
import numpy as np
from pathlib import Path

In [13]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
GROUP = "stressed"
DATA = ROOT / "data" / GROUP

IMAGES_DIR = DATA / "images" / "operculum_images"
LABELS_DIR = DATA / "labels" / "spots"
OUTPUT_DIR = DATA / "images" / "spot_annotations"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
OUTLINE_COLOR = (0, 255, 0)
FILL_ALPHA = 0.25
DRAW_INDEX = True

In [14]:
def read_polygons(label_path, width, height):
    """Return list of (class_id, Nx2 int pixel array) from a YOLO-seg label file."""
    polygons = []
    if not label_path.exists():
        return polygons
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            continue
        cls = int(float(parts[0]))
        coords = np.array(parts[1:], dtype=np.float64).reshape(-1, 2)
        coords[:, 0] *= width
        coords[:, 1] *= height
        polygons.append((cls, np.round(coords).astype(np.int32)))
    return polygons


def annotate(image, polygons):
    h, w = image.shape[:2]
    thickness = max(1, round(min(h, w) / 500))
    font_scale = max(0.4, min(h, w) / 1500)

    overlay = image.copy()
    for _, pts in polygons:
        cv2.fillPoly(overlay, [pts], OUTLINE_COLOR)
    out = cv2.addWeighted(overlay, FILL_ALPHA, image, 1 - FILL_ALPHA, 0)

    for i, (_, pts) in enumerate(polygons, start=1):
        cv2.polylines(out, [pts], isClosed=True, color=OUTLINE_COLOR,
                      thickness=thickness, lineType=cv2.LINE_AA)
        if DRAW_INDEX:
            x, y, bw, bh = cv2.boundingRect(pts)
            org = (x + bw + 2, max(y, 12))
            cv2.putText(out, str(i), org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                        (0, 0, 0), thickness + 2, cv2.LINE_AA)
            cv2.putText(out, str(i), org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                        (255, 255, 255), thickness, cv2.LINE_AA)

    label = f"spots: {len(polygons)}"
    (_, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                     font_scale * 1.5, thickness + 3)
    label_org = (10, 10 + text_h)
    cv2.putText(out, label, label_org, cv2.FONT_HERSHEY_SIMPLEX, font_scale * 1.5,
                (0, 0, 0), thickness + 3, cv2.LINE_AA)
    cv2.putText(out, label, label_org, cv2.FONT_HERSHEY_SIMPLEX, font_scale * 1.5,
                (0, 255, 255), thickness + 1, cv2.LINE_AA)
    return out


def main():
    if not IMAGES_DIR.is_dir():
        raise FileNotFoundError(f"Images folder not found: {IMAGES_DIR.resolve()}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    images = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    done, missing_labels = 0, []

    for img_path in images:
        image = cv2.imdecode(np.fromfile(str(img_path), dtype=np.uint8), cv2.IMREAD_COLOR)
        if image is None:
            print(f"[skip] could not read {img_path.name}")
            continue

        label_path = LABELS_DIR / f"{img_path.stem}.txt"
        if not label_path.exists():
            missing_labels.append(img_path.name)

        h, w = image.shape[:2]
        polygons = read_polygons(label_path, w, h)
        annotated = annotate(image, polygons)

        out_path = OUTPUT_DIR / img_path.name
        ok, buf = cv2.imencode(out_path.suffix, annotated)
        if ok:
            buf.tofile(str(out_path))
            done += 1
        else:
            print(f"[skip] could not encode {img_path.name}")

    print(f"Annotated {done}/{len(images)} images -> {OUTPUT_DIR.resolve()}")
    if missing_labels:
        print(f"{len(missing_labels)} images had no label file: {missing_labels}")

In [15]:
if __name__ == "__main__":
    main()

Annotated 84/84 images -> C:\Users\tala\Downloads\PhD\Kappa Guidelines\newest_work\all_data\stressed\images\spot_annotations
